# 18 — Dual-Corpus Retrieval (AWMF guidelines + MedRAG medical-reference fallback)

**Why this notebook exists.** The AWMF corpus is small and German-specific: it answers questions
about the ten target GP diseases well, but it covers only about a quarter of the generic MedQA
benchmark, so the full-benchmark scores stay low. This notebook keeps the AWMF guidelines as the
primary, authoritative source and adds a large open medical-reference corpus — the **MedRAG
`textbooks` collection** (eighteen standard English medical textbooks, the corpus MedQA itself is
drawn from) — as a fallback. Questions outside the German guideline scope can then be answered
from grounded reference text rather than from the model's own memory.

**What stays untouched.** The AWMF vector store in the managed database is read only, exactly as in
the earlier notebooks. The MedRAG corpus is embedded and indexed **locally inside this session**
(and cached to Drive); nothing is written back to the managed database.

**Pipeline.** question → a HyDE German translation drives the AWMF (German) search, while the
original English question drives the MedRAG (English) search → the two candidate pools are merged
and re-scored by the cross-encoder against the English question → the strongest passages, from
whichever corpus is most relevant, are handed to a strict-grounding generation step. Because both
the embedder (BGE-M3) and the reranker are multilingual, German and English passages compete
fairly.

**Honest expectation.** Adding the reference corpus lifts coverage, so the full-MedQA scores should
rise materially — published MedRAG systems land around 0.7, not 0.9. The corpus-grounded score
over the AWMF-answerable subset remains the measure of the localized guideline behaviour.

**Prerequisites.** Same secrets as before (`NEON_DATABASE_URL`, `OPENROUTER_API_KEY`), a GPU
runtime (the reference corpus is embedded once on the GPU), and enough Drive space for the cached
index (~0.5 GB).

## Installs

In [ ]:
!pip install -q ragas langchain langchain-openai langchain-huggingface psycopg2-binary pgvector langchain-postgres datasets nest_asyncio sentence-transformers sqlalchemy requests rank_bm25

## Vertex patch

A defensive shim so that importing parts of `langchain_community` does not pull in the Vertex
AI SDK, which is not installed in this environment.

In [ ]:
import sys, types
class DummyVertexAI: pass
class DummyChatVertexAI: pass
m = types.ModuleType("langchain_community.llms"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms"] = m
m = types.ModuleType("langchain_community.chat_models"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models"] = m
m = types.ModuleType("langchain_community.chat_models.vertexai"); m.ChatVertexAI = DummyChatVertexAI; sys.modules["langchain_community.chat_models.vertexai"] = m
m = types.ModuleType("langchain_community.llms.vertexai"); m.VertexAI = DummyVertexAI; sys.modules["langchain_community.llms.vertexai"] = m

## Database, dense store, BM25 index, reranker, models

The same infrastructure as the earlier notebooks. The AWMF collection is read from the managed
database and its BM25 index is rebuilt from the same chunks. The answer prompt is a **strict
grounding** prompt: it forbids outside knowledge and asks the model to abstain when the passages
are insufficient — this is what keeps faithfulness high in a clinical setting.

In [ ]:
import os, json, time, random
import pandas as pd, torch, numpy as np, nest_asyncio
from google.colab import userdata, drive
from sqlalchemy import create_engine, text
from langchain_openai import ChatOpenAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_postgres import PGVector
from langchain_core.prompts import PromptTemplate
from sentence_transformers import CrossEncoder
from rank_bm25 import BM25Okapi

nest_asyncio.apply()
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/'
df = pd.read_csv(DRIVE_PATH + 'AWMF_Golden_Dataset_200Q_Final.csv')

NEON = userdata.get('NEON_DATABASE_URL')
os.environ["OPENROUTER_API_KEY"] = userdata.get('OPENROUTER_API_KEY')

engine = create_engine(NEON, pool_pre_ping=True, pool_recycle=180, pool_size=2, max_overflow=2,
                       connect_args={"sslmode":"require","keepalives":1,"keepalives_idle":30})

COLLECTION = "awmf_baseline_bge"

# 1. AWMF dense store (read-only) + query embedder
bge = HuggingFaceEmbeddings(model_name="BAAI/bge-m3", model_kwargs={'device':'cpu'})
vs = PGVector(embeddings=bge, collection_name=COLLECTION, connection=engine, use_jsonb=True)

# 2. AWMF sparse BM25 index
print("Loading AWMF texts for BM25 index...")
with engine.connect() as conn:
    res = conn.execute(text("""SELECT document FROM langchain_pg_embedding
                               WHERE collection_id = (SELECT uuid FROM langchain_pg_collection WHERE name = :c)"""),
                       {"c": COLLECTION})
    all_texts = [r[0] for r in res]
bm25_retriever = BM25Okapi([doc.lower().split(" ") for doc in all_texts])
print(f"AWMF BM25 index built with {len(all_texts)} chunks.")

# 3. Cross-encoder reranker (multilingual: German + English passages compete fairly)
print("Loading reranker...")
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", max_length=512, device="cuda" if torch.cuda.is_available() else "cpu")

def make_llm(model, max_tokens=1024):
    return ChatOpenAI(model=model, api_key=os.environ["OPENROUTER_API_KEY"],
                      base_url="https://openrouter.ai/api/v1", temperature=0,
                      max_tokens=max_tokens, max_retries=6, request_timeout=90)

mistral = make_llm("mistralai/mistral-large")

def safe_invoke(llm, prompt, max_tries=8, base=8):
    for a in range(max_tries):
        try:
            return llm.invoke(prompt).content.strip()
        except Exception as e:
            if a == max_tries-1: raise
            w = min(base*(2**a)+random.uniform(0,3), 120)
            print(f"  retry {a+1}: {str(e)[:70]} ... {w:.0f}s"); time.sleep(w)

# HyDE translation, kept as in the hybrid baseline (drives the German AWMF search).
query_gen_prompt = PromptTemplate(template="""You are a German medical expert.
1. First, precisely TRANSLATE the English question into German.
2. Second, write a 2-sentence hypothetical answer to the question in formal German clinical guideline terminology.
Do NOT use bullet points. Output ONLY the German translation followed directly by the German hypothetical answer.

English Question:
{question}""", input_variables=["question"])

# STRICT grounding answer prompt — the lever that raises faithfulness for a medical system.
qa_prompt = PromptTemplate(template="""You are a clinical assistant. Answer the question in ENGLISH using ONLY the reference passages provided below. The passages may be German clinical guidelines or English medical references.

STRICT RULES:
1. Base every statement directly on the passages. Do NOT use any medical knowledge from outside them.
2. Do NOT add background, mechanisms, or details the passages do not state.
3. If the passages do not contain enough information to answer, reply with exactly: NOT_IN_CORPUS
4. Prefer the source's own wording. Be concise and factual.

REFERENCE PASSAGES:
{context}

QUESTION: {question}

ANSWER (English, grounded strictly in the passages):""", input_variables=["context", "question"])
print("Setup complete.")

## Build the MedRAG reference index (local, cached)

The MedRAG `textbooks` corpus is loaded from Hugging Face, embedded once with BGE-M3 on the GPU,
and stored as a local matrix plus its texts. Both files are cached to Drive, so this heavy step
runs only the first time. The index lives entirely in this session — it is never written to the
managed database.

In [ ]:
from datasets import load_dataset
from sentence_transformers import SentenceTransformer

MEDRAG_EMB = DRIVE_PATH + "medrag_textbooks_bge.npy"
MEDRAG_TXT = DRIVE_PATH + "medrag_textbooks_texts.json"

st = SentenceTransformer("BAAI/bge-m3", device="cuda" if torch.cuda.is_available() else "cpu")

if os.path.exists(MEDRAG_EMB) and os.path.exists(MEDRAG_TXT):
    medrag_emb = np.load(MEDRAG_EMB)
    medrag_texts = json.load(open(MEDRAG_TXT))
    print(f"Loaded cached MedRAG index: {len(medrag_texts)} snippets, dim {medrag_emb.shape[1]}.")
else:
    print("Loading MedRAG 'textbooks' corpus from Hugging Face (first run only)...")
    ds = load_dataset("MedRAG/textbooks", split="train")
    def snippet(r):
        return (r.get("contents") or r.get("content") or "").strip()
    medrag_texts = [snippet(r) for r in ds]
    medrag_texts = [t for t in medrag_texts if len(t) > 20]
    print(f"{len(medrag_texts)} snippets. Embedding with BGE-M3 on the GPU (this is the slow, one-time step)...")
    medrag_emb = st.encode(medrag_texts, batch_size=64, normalize_embeddings=True,
                           show_progress_bar=True, convert_to_numpy=True).astype("float32")
    np.save(MEDRAG_EMB, medrag_emb)
    json.dump(medrag_texts, open(MEDRAG_TXT, "w"))
    print(f"MedRAG index built and cached: {len(medrag_texts)} snippets.")

## Dual-corpus retrieval

Each corpus is searched in its own language. The German HyDE text queries the AWMF guidelines
(dense in the managed store plus BM25), and the original English question queries the MedRAG
textbooks (cosine over the local matrix). The three candidate lists are merged, de-duplicated, and
re-scored by the cross-encoder against the English question, so the final passages are simply the
most relevant ones — from whichever corpus. Each returned passage is tagged with its source, so the
mix can be reported later.

In [ ]:
def medrag_search(query_en, k=20):
    qv = st.encode([query_en], normalize_embeddings=True, convert_to_numpy=True).astype("float32")[0]
    sims = medrag_emb @ qv
    idx = np.argpartition(-sims, min(k, len(sims)-1))[:k]
    idx = idx[np.argsort(-sims[idx])]
    return [medrag_texts[i] for i in idx]

def retrieve(question, k_final=10):
    hyde_de = safe_invoke(mistral, query_gen_prompt.format(question=question))  # German query

    # AWMF (German) candidates
    awmf_dense = [d.page_content for d in vs.as_retriever(search_kwargs={"k": 20}).invoke(hyde_de)]
    awmf_bm25  = bm25_retriever.get_top_n(hyde_de.lower().split(" "), all_texts, n=20)
    awmf_keys  = set(t[:120] for t in (awmf_dense + awmf_bm25))

    # MedRAG (English) candidates
    medrag = medrag_search(question, k=20)

    # merge + de-duplicate
    pool, seen = [], set()
    for t in awmf_dense + awmf_bm25 + medrag:
        key = t[:120]
        if key in seen: continue
        seen.add(key); pool.append(t)

    # rerank against the ENGLISH question (multilingual cross-encoder normalises both corpora)
    scores = reranker.predict([[question, t] for t in pool])
    order = sorted(range(len(pool)), key=lambda i: scores[i], reverse=True)[:k_final]
    ctx = [pool[i] for i in order]
    src = ["AWMF" if c[:120] in awmf_keys else "MedRAG" for c in ctx]
    return ctx, src

# Smoke test
_q = df.iloc[0]['English_Open_Question']
_ctx, _src = retrieve(_q)
print("Question:", _q[:90])
print("Top-k sources:", _src)
print("Top passage:", _ctx[0][:200], "...")

## Generation loop

Every question is answered from the merged top passages with the strict-grounding prompt. The
contexts stored for evaluation are exactly the reranked passages. Results are written to Drive
after each item so a dropped session resumes cleanly.

In [ ]:
gt_df = pd.read_csv(DRIVE_PATH + "AWMF_CorpusGrounded_GroundTruth.csv")
gt_map = gt_df.set_index('English_Open_Question')['corpus_ground_truth'].to_dict()

work = df.reset_index(drop=True)  # full 200 questions
print(f"Generating grounded answers for {len(work)} questions")

out_file = DRIVE_PATH + "DUAL_CORPUS_results.json"
if os.path.exists(out_file):
    res = json.load(open(out_file)); done = set(res["question"])
    print(f"  resuming -- {len(done)} already done")
else:
    res = {"question":[],"answer":[],"contexts":[],"source_mix":[],
           "medqa_ground_truth":[],"corpus_ground_truth":[]}; done = set()

for i in range(len(work)):
    r = work.iloc[i]; q = r['English_Open_Question']
    if q in done: continue
    try:
        ctx, src = retrieve(q)
        ans = safe_invoke(mistral, qa_prompt.format(context="\n\n".join(ctx), question=q))

        res["question"].append(q); res["answer"].append(ans); res["contexts"].append(ctx)
        res["source_mix"].append(src)
        res["medqa_ground_truth"].append(r['English_Correct_Text'])
        res["corpus_ground_truth"].append(gt_map.get(q, "NOT_IN_CORPUS"))
        done.add(q)
        json.dump(res, open(out_file, "w"))

        if len(res["question"]) % 10 == 0: print(f"  {len(res['question'])}/{len(work)}")
        time.sleep(1.5)
    except Exception as e:
        print("err", i, str(e)[:100]); time.sleep(8)

print(f"Generation complete -> {out_file}")

## Evaluation

The same GPT-4o-mini judge and the same four RAGAS metrics as every earlier notebook, so the
numbers stay directly comparable. Two views are reported: the **full 200** MedQA questions (where
the reference fallback is expected to lift coverage) and the **corpus-grounded** answerable subset
(the localized German-guideline behaviour). A source-mix line shows how often the reference corpus
was actually used.

In [ ]:
from datasets import Dataset, Features, Value, Sequence
from ragas import evaluate
from ragas.metrics import context_precision, context_recall, faithfulness, answer_relevancy
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.run_config import RunConfig

judge = LangchainLLMWrapper(make_llm("openai/gpt-4o-mini", max_tokens=4096))
remb = LangchainEmbeddingsWrapper(bge)
feat = Features({"question":Value("string"),"answer":Value("string"),
                 "contexts":Sequence(Value("string")),"ground_truth":Value("string")})

def score(path, gt_key, keep=None):
    d = json.load(open(path))
    idx = [i for i in range(len(d["question"])) if keep is None or keep(d, i)]
    dd = {"question":[d["question"][i] for i in idx],
          "answer":[d["answer"][i] for i in idx],
          "contexts":[d["contexts"][i] for i in idx],
          "ground_truth":[d[gt_key][i] for i in idx]}
    ds = Dataset.from_dict(dd, features=feat)
    out = evaluate(ds, metrics=[context_precision, context_recall, faithfulness, answer_relevancy],
                   llm=judge, embeddings=remb, run_config=RunConfig(timeout=300, max_workers=2, max_retries=5))
    return len(idx), dict(out.to_pandas()[["context_precision","context_recall","faithfulness","answer_relevancy"]].mean().round(3))

in_corpus = lambda d, i: d["corpus_ground_truth"][i] not in (None, "", "NOT_IN_CORPUS")

# Source mix: what fraction of the top passages came from the MedRAG fallback
_d = json.load(open(out_file))
_flat = [s for row in _d["source_mix"] for s in row]
_medrag = sum(1 for s in _flat if s == "MedRAG")
print(f"Source mix across all retrieved passages: MedRAG {100*_medrag/max(len(_flat),1):.0f}% / AWMF {100*(len(_flat)-_medrag)/max(len(_flat),1):.0f}%\n")

print("=== DUAL-CORPUS SCORES ===")
n, s = score(out_file, "medqa_ground_truth");                  print(f"Full 200 (MedQA GT, n={n}):        ", s)
n, s = score(out_file, "corpus_ground_truth", keep=in_corpus); print(f"Corpus-Grounded (answerable, n={n}):", s)